# 🖼️ HyMPOR — Full Restoration Pipeline
### Bringing Old Photos → GFPGAN → SAM2 → AOT-GAN → DeOldify

> ✅ تأكد من تفعيل GPU: **Runtime ▸ Change runtime type ▸ T4 GPU**

---
## 📋 ترتيب التشغيل

| الخطوة | الوصف | متى تشغّلها؟ |
|--------|--------|---------------|
| **خلية 0** | تنزيل المشروع كاملاً من GitHub Release وفكّه | **مرة واحدة فقط** في أول استخدام |
| **خلايا 1–4** | تهيئة المودلات وإعداد البيئة | **مرة واحدة** لكل جلسة |
| **خلايا 5–11** | معالجة الصور | **كرر** لكل صورة |

> ⚠️ **ملاحظة:** الخلية 0 تنزّل ~10 جيجا — قد تستغرق 10–20 دقيقة حسب سرعة الاتصال.
> لا حاجة لأي روابط خارجية — المشروع كامل بالأوزان داخل الـ Release.


## ═══════════════════════════════════════
## ⬇️ الخلية 0 — تنزيل المشروع الكامل (أوزان + كود)
## شغّل مرة واحدة فقط في أول استخدام
## ═══════════════════════════════════════
> المشروع مقسّم إلى **6 أجزاء RAR** في GitHub Release.  
> هذه الخلية تنزّلها، تتحقق من SHA256، ثم تفكّها تلقائياً في `/content/hybrid_project`.


In [ ]:
import os, subprocess, requests

RELEASE_BASE = 'https://github.com/proanas/HyMPOR/releases/download/v1.0.0'
DEST         = '/content'
ROOT         = '/content/hybrid_project'

# ── يجلب أحجام الملفات من GitHub API تلقائياً ──────────────────────────────
print('🔍 جلب معلومات الـ Release من GitHub...')
api = requests.get(
    'https://api.github.com/repos/proanas/HyMPOR/releases/tags/v1.0.0',
    headers={'Accept': 'application/vnd.github+json'}
).json()

PARTS = {a['name']: a['size'] for a in api['assets'] if a['name'].endswith('.rar')}
print(f'✅ {len(PARTS)} أجزاء في الـ Release: {list(PARTS.keys())}')

if os.path.exists(ROOT):
    print(f'✅ المشروع موجود مسبقاً — تخطي التنزيل')
    print(f'   لإعادة التنزيل: !rm -rf {ROOT}')
else:
    subprocess.run('apt-get install -qq unrar', shell=True, check=True)
    print('✅ unrar جاهز\n')

    all_ok = True
    for fname, expected_size in sorted(PARTS.items()):
        path = os.path.join(DEST, fname)
        if os.path.exists(path):
            actual_size = os.path.getsize(path)
            if actual_size == expected_size:
                print(f'✔️  موجود وسليم: {fname} ({actual_size/1024**3:.2f} GB)')
                continue
            else:
                print(f'⚠️  {fname} ناقص ({actual_size/1024**3:.2f} GB من {expected_size/1024**3:.2f} GB) — إعادة التنزيل')
                os.remove(path)

        url = f'{RELEASE_BASE}/{fname}'
        print(f'⬇️  {fname} ({expected_size/1024**3:.2f} GB)')
        ret = os.system(f'wget -q --show-progress "{url}" -O "{path}"')
        if ret != 0 or os.path.getsize(path) != expected_size:
            print(f'❌ فشل تنزيل {fname}')
            all_ok = False

    if not all_ok:
        raise RuntimeError('❌ بعض الأجزاء فشلت — أعد تشغيل الخلية')

    print('\n📂 جاري فك الضغط...')
    ret = os.system(f'unrar x -o+ "{DEST}/hybrid_project.part1.rar" "{DEST}/"')
    if ret != 0:
        raise RuntimeError('❌ فشل فك الضغط')

    print('🧹 حذف ملفات RAR...')
    for fname in PARTS:
        p = os.path.join(DEST, fname)
        if os.path.exists(p):
            os.remove(p)

assert os.path.exists(ROOT), f'❌ لم يُنشأ {ROOT}'
print(f'\n🎉 المشروع جاهز: {ROOT}')
print(f'   المحتوى: {os.listdir(ROOT)}')

---
## ═══════════════════════════════════════
## 🔵 قسم التهيئة — شغّل مرة واحدة فقط في بداية الجلسة
## ═══════════════════════════════════════


## الخطوة 1 — إعداد المسارات

In [ ]:
import os, sys

ROOT         = '/content/hybrid_project'
REPO_PATH    = f'{ROOT}/modules/Bringing_Old_Photos_Back_to_Life'
SAM2_ROOT    = f'{ROOT}/modules/SAM2'
AOT_SRC      = f'{ROOT}/modules/AOT_GAN/src'
GFPGAN_DIR   = f'{ROOT}/modules/GFPGAN'
DEOLDIFY_DIR = f'{ROOT}/modules/DeOldify'

assert os.path.exists(ROOT), '❌ شغّل الخلية 0 أولاً لتنزيل المشروع'
print('✅ المسارات جاهزة')
print(f'   ROOT      : {ROOT}')
print(f'   BOPBL     : {REPO_PATH}')
print(f'   GFPGAN    : {GFPGAN_DIR}')
print(f'   DeOldify  : {DEOLDIFY_DIR}')

## الخطوة 2 — إعداد البيئة وتحميل pipeline

1.   List item
2.   List item

(SAM2 + AOT-GAN)

In [ ]:
# حل المشكلة
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
!pip install -q opencv-contrib-python

In [ ]:
import os, sys, subprocess, torch

# ─── 1. تثبيت المكتبات ─────────────────────────────────────────────────────
print('📦 تثبيت المكتبات...')
os.system('pip install -q tensorboardX dlib scikit-image iopath hydra-core')
print('✅ المكتبات جاهزة')

# ─── 2. إضافة مسارات SAM2 و AOT-GAN ───────────────────────────────────────
for p in [SAM2_ROOT, AOT_SRC, ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
print('✅ مسارات SAM2 و AOT-GAN جاهزة')

# ─── 3. إصلاح مسار detection.py ───────────────────────────────────────────
det_file = f'{REPO_PATH}/Global/detection.py'
with open(det_file, 'r') as f:
    content = f.read()
old_det = 'checkpoint_path = os.path.join(os.path.dirname(__file__), "checkpoints/detection/FT_Epoch_latest.pt")'
new_det = f'checkpoint_path = "{REPO_PATH}/Global/checkpoints/detection/FT_Epoch_latest.pt"'
if old_det in content:
    content = content.replace(old_det, new_det)
    with open(det_file, 'w') as f:
        f.write(content)
print('✅ detection.py جاهز')

# ─── 4. إصلاح مسار test.py ─────────────────────────────────────────────────
test_file = f'{REPO_PATH}/Global/test.py'
with open(test_file, 'r') as f:
    content = f.read()
old_test = '"./checkpoints/restoration"'
new_test = f'"{REPO_PATH}/Global/checkpoints/restoration"'
if old_test in content:
    content = content.replace(old_test, new_test)
    with open(test_file, 'w') as f:
        f.write(content)
print('✅ test.py جاهز')

# ─── 5. إصلاح numpy casting ────────────────────────────────────────────────
align_file = f'{REPO_PATH}/Face_Detection/align_warp_back_multiple_dlib.py'
with open(align_file, 'r') as f:
    content = f.read()
if 'mask *= 255.0' in content:
    content = content.replace('mask *= 255.0', 'mask = mask.astype(np.float64) * 255.0')
    with open(align_file, 'w') as f:
        f.write(content)
print('✅ إصلاح numpy جاهز')

# ─── 6. تحميل SAM2+AOT-GAN pipeline ───────────────────────────────────────
from pipeline.config import print_config
from pipeline.run import HybridPipeline
print_config()
pipeline = HybridPipeline()

os.chdir(REPO_PATH)
print('\n🎉 Pipeline SAM2 + AOT-GAN جاهز!')

## الخطوة 3 — تحميل GFPGAN
> شغّل مرة واحدة فقط — يبقى في الذاكرة لكل الصور


In [ ]:
import sys, glob, os, torch

GFPGAN_WEIGHTS = f'{GFPGAN_DIR}/experiments/pretrained_models/GFPGANv1.4.pth'

os.system('pip install -q basicsr facexlib realesrgan')

# إصلاح basicsr مع torchvision الحديث
for _f in glob.glob('/usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py'):
    txt = open(_f).read()
    if 'functional_tensor' in txt:
        open(_f, 'w').write(txt.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'))
        print(f'✅ إصلاح basicsr: {_f}')

if GFPGAN_DIR not in sys.path:
    sys.path.insert(0, GFPGAN_DIR)

if not os.path.exists(GFPGAN_WEIGHTS):
    raise FileNotFoundError(f'❌ الأوزان غير موجودة: {GFPGAN_WEIGHTS}')
print(f'✅ الأوزان موجودة: {GFPGAN_WEIGHTS}')

from gfpgan import GFPGANer
torch.cuda.empty_cache()

gfpganer = GFPGANer(
    model_path         = GFPGAN_WEIGHTS,
    upscale            = 2,
    arch               = 'clean',
    channel_multiplier = 2,
    bg_upsampler       = None
)
print('✅ GFPGAN محمّل وجاهز لكل الصور')

try:
    from pipeline.selector import set_external_face_helper
    set_external_face_helper(gfpganer.face_helper)
except Exception as _e:
    print(f'ℹ️  selector سيحمّل كاشفه الخاص لاحقاً: {_e}')

## الخطوة 4 — تحميل DeOldify
> شغّل مرة واحدة فقط — يبقى في الذاكرة لكل الصور


In [ ]:
import sys, os, shutil, subprocess, collections, collections.abc, torch as _torch

DEOLDIFY_DIR  = f'{ROOT}/modules/DeOldify'
STABLE_WEIGHT = f'{DEOLDIFY_DIR}/models/ColorizeStable_gen.pth'
TORCH_CACHE   = f'{DEOLDIFY_DIR}/torch_cache'
PKG_DIR       = f'{DEOLDIFY_DIR}/site_packages'

assert os.path.exists(STABLE_WEIGHT), f'❌ الأوزان غير موجودة: {STABLE_WEIGHT}'

os.environ['TORCH_HOME'] = TORCH_CACHE
os.makedirs(f'{TORCH_CACHE}/hub/checkpoints', exist_ok=True)

os.chdir(DEOLDIFY_DIR)
for _p in [PKG_DIR, DEOLDIFY_DIR]:
    if _p in sys.path: sys.path.remove(_p)
sys.path.insert(0, PKG_DIR)
sys.path.insert(0, DEOLDIFY_DIR)

_CONFLICT = ('torch', 'torchvision', 'torchaudio', 'fastai')
if os.path.exists(PKG_DIR):
    for _item in os.listdir(PKG_DIR):
        if any(_item.lower().startswith(_p) for _p in _CONFLICT):
            _full = os.path.join(PKG_DIR, _item)
            shutil.rmtree(_full) if os.path.isdir(_full) else os.remove(_full)
os.makedirs(PKG_DIR, exist_ok=True)

for _name in ('Sized','Callable','Mapping','MutableMapping','Iterable',
              'Iterator','Sequence','MutableSequence','Set','MutableSet'):
    if not hasattr(collections, _name):
        setattr(collections, _name, getattr(collections.abc, _name))

if not hasattr(_torch, '_orig_load_backup'):
    _torch._orig_load_backup = _torch.load

def _safe_load(f, map_location=None, pickle_module=None, weights_only=None, **kw):
    return _torch._orig_load_backup(f, map_location=map_location, weights_only=False, **kw)

_torch.load = _safe_load

from deoldify import device as _dev
from deoldify.device_id import DeviceId
_dev.set(device=DeviceId.GPU0)

_PKGS = {'ffmpeg': 'ffmpeg-python', 'yt_dlp': 'yt-dlp', 'tensorboardX': 'tensorboardX'}
for _mod, _pkg in _PKGS.items():
    try:
        __import__(_mod)
    except ImportError:
        subprocess.run(
            f'{sys.executable} -m pip install -q "{_pkg}" --target "{PKG_DIR}" --no-deps',
            shell=True, capture_output=True)
        print(f'  📦 {_pkg}')

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*?Your .*? set is empty.*?')
warnings.filterwarnings('ignore', category=FutureWarning)

# هنا تم تصليح مشاكل توافق المكتبات
# تعطيل فحص pkg_resources لتفادي تعارض إصدارات لا علاقة له بعمل DeOldify
import pkg_resources
pkg_resources.require = lambda *a, **k: None

import fastai
from deoldify.visualize import *

#----
import fastai
from deoldify.visualize import *
_torch.backends.cudnn.benchmark = True
_torch.cuda.empty_cache()

colorizer = get_image_colorizer(artistic=False)
print('✅ DeOldify Stable محمّل وجاهز لكل الصور')

os.chdir(REPO_PATH)


---
## ═══════════════════════════════════════
## 🔁 قسم المعالجة — كرّر من هنا لكل صورة
## ═══════════════════════════════════════


## الخطوة 5 — رفع الصورة

In [ ]:
from google.colab import files
from PIL import Image
import os

upload_path = os.path.join(REPO_PATH, 'test_images', 'upload')
os.makedirs(upload_path, exist_ok=True)

for f in os.listdir(upload_path):
    os.remove(os.path.join(upload_path, f))

uploaded = files.upload()
for filename, data in uploaded.items():
    save_path = os.path.join(upload_path, filename)
    with open(save_path, 'wb') as f:
        f.write(data)
    with Image.open(save_path) as img:
        print(f'✅ تم رفع: {filename}  ({img.size[0]}×{img.size[1]} px, {img.mode})')

## الخطوة 6 — تصغير الصورة (اختياري)
> **غيّر `MAX_SIZE`** حسب حاجتك:
> - `512` → سريع، مناسب لـ T4
> - `768` → جودة أفضل، يحتاج ذاكرة أكبر
> - `0`   → **تعطيل التصغير كلياً**


In [ ]:
from PIL import Image
import os

MAX_SIZE = 512  # ← غيّر هنا | 0 = تعطيل التصغير

if MAX_SIZE == 0:
    print('ℹ️  التصغير معطّل — ستُعالَج الصورة بحجمها الأصلي')
    for filename in os.listdir(upload_path):
        try:
            with Image.open(os.path.join(upload_path, filename)) as img:
                print(f'  📐 {filename}: {img.size[0]}×{img.size[1]} px')
        except Exception:
            pass
else:
    for filename in os.listdir(upload_path):
        filepath = os.path.join(upload_path, filename)
        try:
            img = Image.open(filepath)
            w, h = img.size
            if w > MAX_SIZE or h > MAX_SIZE:
                ratio  = MAX_SIZE / max(w, h)
                new_w  = int(w * ratio)
                new_h  = int(h * ratio)
                img    = img.resize((new_w, new_h), Image.LANCZOS)
                img.save(filepath)
                print(f'✅ {filename}: {w}×{h} → {new_w}×{new_h}')
            else:
                print(f'✔️  {filename}: {w}×{h} — لا يحتاج تصغير')
        except Exception as e:
            print(f'❌ {filename}: {e}')

## الخطوة 7 — المرحلة الأولى: Bringing Old Photos

In [ ]:
import torch, shutil
torch.cuda.empty_cache()

os.chdir(REPO_PATH)
output_path = os.path.join(REPO_PATH, 'upload_output')

if os.path.exists(output_path):
    shutil.rmtree(output_path)
    print('🧹 تم تنظيف مخرجات المعالجة السابقة')
os.makedirs(output_path, exist_ok=True)

cmd = f"""python run.py \\
  --input_folder "{upload_path}" \\
  --output_folder "{output_path}" \\
  --GPU 0 \\
  --with_scratch 2>&1"""

print('🚀 المرحلة الأولى: Bringing Old Photos...')
result = os.popen(cmd).read()

_SKIP_PREFIXES = ('you are processing', 'processing: ')
filtered_lines = [
    line for line in result.split('\n')
    if not line.strip().lower().startswith(_SKIP_PREFIXES)
]
print('\n'.join(filtered_lines))

final_output = os.path.join(output_path, 'final_output')
stage1_results = sorted([
    os.path.join(final_output, f)
    for f in os.listdir(final_output)
    if os.path.splitext(f)[0] in {os.path.splitext(x)[0] for x in os.listdir(upload_path)}
])
print(f'✅ المرحلة الأولى انتهت — {len(stage1_results)} صورة جاهزة للمرحلة الثانية')
for p in stage1_results:
    print(f'  📄 {os.path.basename(p)}')

import cv2, matplotlib.pyplot as plt
upload_files = sorted(os.listdir(upload_path))
for i, (orig_name, res_path) in enumerate(zip(upload_files, stage1_results)):
    orig_path = os.path.join(upload_path, orig_name)
    orig = cv2.cvtColor(cv2.imread(orig_path), cv2.COLOR_BGR2RGB)
    rest = cv2.cvtColor(cv2.imread(res_path),  cv2.COLOR_BGR2RGB)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.imshow(orig); ax1.set_title(f'Original: {orig_name}', fontsize=12); ax1.axis('off')
    ax2.imshow(rest); ax2.set_title(f'After Bringing Old Photos', fontsize=12); ax2.axis('off')
    plt.suptitle(f'image {i+1} From {len(stage1_results)}', fontsize=13)
    plt.tight_layout()
    plt.show()


## الخطوة 8 — المرحلة الثانية: SAM2 + AOT-GAN
### ارسم على منطقة التلف الكبير المتبقي ثم اضغط تأكيد

> 💡 إذا لديك أكثر من صورة، غيّر `IMAGE_INDEX` ثم شغّل الخلية مجدداً لكل صورة.


In [ ]:
import base64, io as _io, numpy as np
from PIL import Image
from IPython.display import display, HTML
from google.colab import output
import matplotlib.pyplot as plt

IMAGE_INDEX = 0  # ← غيّر هنا إذا رفعت أكثر من صورة (0=الأولى، 1=الثانية...)

if IMAGE_INDEX >= len(stage1_results):
    raise IndexError(f'❌ IMAGE_INDEX={IMAGE_INDEX} خارج النطاق — عدد الصور: {len(stage1_results)}')

selected_image_path = stage1_results[IMAGE_INDEX]
print(f'📷 الصورة المحددة: {os.path.basename(selected_image_path)}')

torch.cuda.empty_cache()
image_np = pipeline.load_image(selected_image_path)
H, W = image_np.shape[:2]

buf = _io.BytesIO()
Image.fromarray(image_np).save(buf, format='PNG')
b64 = base64.b64encode(buf.getvalue()).decode()

def on_mask_received(draw_b64):
    global last_result_path
    try:
        import base64 as _b64, io as _io2, numpy as _np2
        from PIL import Image as _Img2
        import matplotlib.pyplot as _plt
        from IPython.display import display as _disp
        _mask = _np2.array(
            _Img2.open(_io2.BytesIO(_b64.b64decode(draw_b64))).convert('L'))
        if _mask.max() < 10:
            last_result_path = selected_image_path
            print('ℹ️ لا يوجد رسم — تم تخطي AOT-GAN')
            _img_prev = _np2.array(_Img2.open(last_result_path).convert('RGB'))
            _fig, _ax = _plt.subplots(figsize=(8,6))
            _ax.imshow(_img_prev)
            _ax.set_title('الصورة جاهزة للمرحلة التالية (GFPGAN)', fontsize=13)
            _ax.axis('off')
            _plt.tight_layout(); _disp(_fig); _plt.close(_fig)
            print(f'✅ جاهز: {os.path.basename(last_result_path)}')
            return
        print('\n⚙️ جاري المعالجة...')
        result = pipeline.process_drawn_mask(draw_b64)
        pipeline.show_result(result)
        last_result_path = result['out_img']
        print(f'\n✅ النتيجة محفوظة: {last_result_path}')
    except Exception as e:
        print(f'❌ خطأ: {e}')
        import traceback; traceback.print_exc()

output.register_callback('process_mask', on_mask_received)
last_result_path = None

display(HTML(f'''
<div style="position:relative;display:inline-block;
            border:2px solid #555;border-radius:8px;overflow:hidden">
  <img id="bg" src="data:image/png;base64,{b64}"
       style="display:block;max-width:600px;width:100%">
  <canvas id="cv"
       style="position:absolute;top:0;left:0;
              width:100%;height:100%;opacity:0.55;cursor:crosshair">
  </canvas>
</div>
<div style="margin-top:8px;display:flex;gap:10px;align-items:center;flex-wrap:wrap">
  <label>حجم الفرشاة:
    <input id="brush" type="range" min="3" max="80" value="10"
           style="width:120px">
  </label>
  <button onclick="clearCanvas()"
    style="padding:6px 14px;background:#e74c3c;color:#fff;
           border:none;border-radius:6px;cursor:pointer">🗑️ مسح</button>
  <button onclick="exportMask()"
    style="padding:6px 14px;background:#27ae60;color:#fff;
           border:none;border-radius:6px;cursor:pointer">
    ✅ تأكيد ← SAM2 + AOT-GAN
  </button>
  <span id="status" style="color:#888;font-size:13px"></span>
</div>
<script>
const bg=document.getElementById("bg"),
      cv=document.getElementById("cv"),
      ctx=cv.getContext("2d");
let painting=false;
function init(){{cv.width=bg.clientWidth;cv.height=bg.clientHeight;
  ctx.fillStyle="black";ctx.fillRect(0,0,cv.width,cv.height);}}
bg.onload=init;
if(bg.complete) init();
function pos(e){{const r=cv.getBoundingClientRect();
  const touch=e.touches?e.touches[0]:e;
  return[touch.clientX-r.left,touch.clientY-r.top];}}
function draw(e){{if(!painting)return;
  const[x,y]=pos(e),r=+document.getElementById("brush").value;
  ctx.beginPath();ctx.arc(x,y,r,0,Math.PI*2);
  ctx.fillStyle="white";ctx.fill();}}
cv.addEventListener("mousedown",e=>{{painting=true;draw(e)}});
cv.addEventListener("mousemove",draw);
cv.addEventListener("mouseup",()=>painting=false);
cv.addEventListener("mouseleave",()=>painting=false);
cv.addEventListener("touchstart",e=>{{painting=true;draw(e);e.preventDefault();}},{{passive:false}});
cv.addEventListener("touchmove",e=>{{draw(e);e.preventDefault();}},{{passive:false}});
cv.addEventListener("touchend",()=>painting=false);
function clearCanvas(){{ctx.fillStyle="black";ctx.fillRect(0,0,cv.width,cv.height);
  document.getElementById("status").textContent="تم المسح";}}
function exportMask(){{document.getElementById("status").textContent="⏳ جاري الإرسال...";
  const data=cv.toDataURL("image/png").split(",")[1];
  google.colab.kernel.invokeFunction("process_mask",[data],{{}});
  document.getElementById("status").textContent="✅ تم الإرسال"; }}
</script>
'''))


## الخطوة 9 — المرحلة الثالثة: GFPGAN
> النموذج محمّل مسبقاً — هذه الخطوة للترميم فقط


In [ ]:
import cv2, matplotlib.pyplot as plt

if not last_result_path or not os.path.exists(last_result_path):
    if 'selected_image_path' in dir() and os.path.exists(selected_image_path):
        last_result_path = selected_image_path
        print('ℹ️ AOT-GAN تم تخطيه — GFPGAN يعمل على نتيجة المرحلة الأولى')
    elif 'stage1_results' in dir() and stage1_results:
        last_result_path = stage1_results[0]
        print('ℹ️ فولباك: GFPGAN على أول نتيجة من المرحلة الأولى')
    else:
        raise ValueError('❌ شغّل الخطوة 7 (Bringing Old Photos) أولاً')

torch.cuda.empty_cache()
input_bgr = cv2.imread(last_result_path, cv2.IMREAD_COLOR)
_, restored_faces, restored_img = gfpganer.enhance(
    input_bgr,
    has_aligned      = False,
    only_center_face = False,
    paste_back       = True,
    weight           = 0.5
)

OUT_DIR = '/content/gfpgan_results'
os.makedirs(OUT_DIR, exist_ok=True)
base        = os.path.splitext(os.path.basename(last_result_path))[0]
gfpgan_path = f'{OUT_DIR}/{base}_GFPGAN.png'
out_img     = restored_img if restored_img is not None else input_bgr
cv2.imwrite(gfpgan_path, out_img)
print(f'✅ النتيجة محفوظة: {gfpgan_path}')
print(f'   عدد الوجوه المُرمَّمة: {len(restored_faces)}')

if restored_faces:
    faces_dir = f'{ROOT}/dataset/output/faces_{base}'
    os.makedirs(faces_dir, exist_ok=True)
    for _fi, _face in enumerate(restored_faces):
        cv2.imwrite(f'{faces_dir}/face_{_fi+1:02d}.png', _face)
    print(f'   الوجوه بدقة عالية: {faces_dir}')

last_result_path = gfpgan_path

before = cv2.cvtColor(input_bgr, cv2.COLOR_BGR2RGB)
after  = cv2.cvtColor(out_img,   cv2.COLOR_BGR2RGB)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
ax1.imshow(before); ax1.set_title('After AOT-GAN',   fontsize=14); ax1.axis('off')
ax2.imshow(after);  ax2.set_title('After GFPGAN',    fontsize=14); ax2.axis('off')
plt.tight_layout(); plt.show()

if restored_faces:
    fig2, axes = plt.subplots(1, len(restored_faces), figsize=(5*len(restored_faces), 5))
    if len(restored_faces) == 1: axes = [axes]
    for ax, face in zip(axes, restored_faces):
        ax.imshow(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))
        ax.set_title('Face Enhancement', fontsize=12); ax.axis('off')
    plt.tight_layout(); plt.show()


## الخطوة 10 — المرحلة الرابعة: DeOldify
> يفحص تلقائياً: صورة ملونة → تخطي | أبيض وأسود → تلوين

> 💡 **`RENDER_FACTOR`**: كلما زاد، جودة أعلى لكن أبطأ.  
> القيم المقترحة: `20` (سريع) · `30` (متوازن) · `40` (أفضل جودة)


In [ ]:
RENDER_FACTOR = 20  # ← غيّر هنا (20-40)

import cv2, numpy as np, os, shutil

_img_check = cv2.imread(last_result_path)
_b, _g, _r = cv2.split(_img_check)
_is_bw = (np.std(_r.astype(int) - _g.astype(int)) < 15 and
          np.std(_r.astype(int) - _b.astype(int)) < 15)

print(f'📊 فحص الصورة: {"أبيض وأسود ← سيتم التلوين" if _is_bw else "ملونة ← تخطي DeOldify"}')

if not _is_bw:
    print('✅ الصورة ملونة — جاهزة للتنزيل')
else:
    os.chdir(DEOLDIFY_DIR)
    RENDER_FACTOR = 20
    print(f'🎨 جاري التلوين (render_factor={RENDER_FACTOR})...')
    _result_path = colorizer.plot_transformed_image(
        path          = last_result_path,
        render_factor = RENDER_FACTOR,
        compare       = True,
        watermarked   = False
    )
    show_image_in_notebook(_result_path)
    _deo_dir = '/content/deoldify_results'
    os.makedirs(_deo_dir, exist_ok=True)
    _deo_base  = os.path.splitext(os.path.basename(last_result_path))[0]
    _colorized = f'{_deo_dir}/{_deo_base}_colorized.png'
    shutil.copy(str(_result_path), _colorized)
    last_result_path = _colorized
    print(f'✅ النتيجة الملوّنة: {_colorized}')
    os.chdir(REPO_PATH)


## الخطوة 11 — تنزيل النتيجة النهائية

In [ ]:
from google.colab import files
import cv2, matplotlib.pyplot as plt, os

if not last_result_path or not os.path.exists(last_result_path):
    print('⚠️ لا توجد نتيجة بعد — شغّل الخطوة 8 أولاً')
else:
    # مقارنة الأصل بالنتيجة النهائية
    orig_files = sorted(os.listdir(upload_path))
    if orig_files:
        orig_img  = cv2.cvtColor(cv2.imread(os.path.join(upload_path, orig_files[0])), cv2.COLOR_BGR2RGB)
        final_img = cv2.cvtColor(cv2.imread(last_result_path), cv2.COLOR_BGR2RGB)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        ax1.imshow(orig_img);  ax1.set_title('📷 الصورة الأصلية',  fontsize=14); ax1.axis('off')
        ax2.imshow(final_img); ax2.set_title('✨ النتيجة النهائية', fontsize=14); ax2.axis('off')
        plt.suptitle('HyMPOR — مقارنة الأصل بالنتيجة', fontsize=15, fontweight='bold')
        plt.tight_layout(); plt.show()
    files.download(last_result_path)
    print(f'✅ جاري تنزيل: {os.path.basename(last_result_path)}')